In [0]:
silver_df = spark.table("silver_nba_shot_logs_cleaned")

display(silver_df.limit(5))

In [0]:
from pyspark.sql.functions import col, when, upper, trim

gold_df = (
    silver_df

    # Shot distance buckets
    .withColumn("is_close_shot", when(col("shot_dist") <= 8, 1).otherwise(0))
    .withColumn("is_mid_range_shot", when((col("shot_dist") > 8) & (col("shot_dist") < 23.75), 1).otherwise(0))
    .withColumn("is_three_point_shot", when((col("shot_dist") >= 23.75) & (col("shot_dist") < 28), 1).otherwise(0))
    .withColumn("is_deep_shot", when(col("shot_dist") >= 28, 1).otherwise(0))

    # Defender distance buckets
    .withColumn("is_very_tight_defense", when(col("close_def_dist") <= 2, 1).otherwise(0))
    .withColumn("is_tight_defense", when((col("close_def_dist") > 2) & (col("close_def_dist") <= 4), 1).otherwise(0))
    .withColumn("is_open_defense", when((col("close_def_dist") > 4) & (col("close_def_dist") <= 6), 1).otherwise(0))
    .withColumn("is_wide_open_defense", when(col("close_def_dist") > 6, 1).otherwise(0))

    # Game/time context
    .withColumn("is_last_5_min_game", when(col("total_seconds_left_in_game") <= 300, 1).otherwise(0))
    .withColumn("is_last_2_min_period", when(col("game_clock_seconds_remaining") <= 120, 1).otherwise(0))
    .withColumn("is_end_of_period", when(col("game_clock_seconds_remaining") <= 24, 1).otherwise(0))

    # Shot clock context
    .withColumn("is_very_late_clock", when(col("shot_clock_adjusted") <= 4, 1).otherwise(0))
    .withColumn("is_late_clock", when((col("shot_clock_adjusted") > 4) & (col("shot_clock_adjusted") <= 7), 1).otherwise(0))

    # Home / away context
    .withColumn("is_home_game", when(upper(trim(col("location"))) == "H", 1).otherwise(0))

    # Shot creation style
    .withColumn("is_catch_and_shoot", when((col("touch_time") <= 2) & (col("dribbles") == 0), 1).otherwise(0))
)

In [0]:
feature_cols = [
    # Identifiers / context
    "game_id",
    "player_name",
    "player_id",
    "period",
    "location",

    # Raw numeric features
    "game_clock_seconds_remaining",
    "total_seconds_left_in_game",
    "shot_clock",
    "shot_clock_adjusted",
    "is_shot_clock_missing",
    "dribbles",
    "touch_time",
    "shot_dist",
    "pts_type",
    "close_def_dist",

    # Shot distance features
    "is_close_shot",
    "is_mid_range_shot",
    "is_three_point_shot",
    "is_deep_shot",

    # Defender distance features
    "is_very_tight_defense",
    "is_tight_defense",
    "is_open_defense",
    "is_wide_open_defense",

    # Game/time context features
    "is_last_5_min_game",
    "is_last_2_min_period",
    "is_end_of_period",

    # Shot clock features
    "is_very_late_clock",
    "is_late_clock",

    # Extra features
    "is_home_game",
    "is_catch_and_shoot",

    # Target
    "label"
]

gold_model_df = gold_df.select(*feature_cols)

display(gold_model_df.limit(10))

In [0]:
spark.sql("DROP TABLE IF EXISTS gold_nba_shot_quality_features")
gold_model_df.write.mode("overwrite").saveAsTable("gold_nba_shot_quality_features")

In [0]:
spark.table("gold_nba_shot_quality_features").printSchema()

In [0]:
# Overall Summary
spark.sql("""
SELECT 
  COUNT(*) AS rows,
  AVG(label) AS make_rate,

  AVG(shot_dist) AS avg_shot_dist,
  AVG(close_def_dist) AS avg_close_def_dist,
  AVG(shot_clock_adjusted) AS avg_shot_clock_adjusted,
  AVG(game_clock_seconds_remaining) AS avg_game_clock_seconds_remaining,
  AVG(total_seconds_left_in_game) AS avg_total_seconds_left_in_game,

  AVG(is_shot_clock_missing) AS shot_clock_missing_rate,
  AVG(is_close_shot) AS close_shot_rate,
  AVG(is_mid_range_shot) AS mid_range_shot_rate,
  AVG(is_three_point_shot) AS three_point_shot_rate,
  AVG(is_deep_shot) AS deep_shot_rate,

  AVG(is_very_tight_defense) AS very_tight_defense_rate,
  AVG(is_tight_defense) AS tight_defense_rate,
  AVG(is_open_defense) AS open_defense_rate,
  AVG(is_wide_open_defense) AS wide_open_defense_rate,

  AVG(is_last_5_min_game) AS last_5_min_game_rate,
  AVG(is_last_2_min_period) AS last_2_min_period_rate,
  AVG(is_end_of_period) AS end_of_period_rate,

  AVG(is_very_late_clock) AS very_late_clock_rate,
  AVG(is_late_clock) AS late_clock_rate,

  AVG(is_home_game) AS home_game_rate,
  AVG(is_catch_and_shoot) AS catch_and_shoot_rate
FROM gold_nba_shot_quality_features
""").show()

In [0]:
# Make Rates by Shot Distance
spark.sql("""
SELECT 
  CASE
    WHEN is_close_shot = 1 THEN 'Close: <= 8 ft'
    WHEN is_mid_range_shot = 1 THEN 'Mid-range: >8 to <23.75 ft'
    WHEN is_three_point_shot = 1 THEN 'Three-point: >=23.75 to <28 ft'
    WHEN is_deep_shot = 1 THEN 'Deep: >=28 ft'
    ELSE 'Unknown'
  END AS shot_distance_bucket,
  COUNT(*) AS shots,
  AVG(label) AS make_rate,
  AVG(shot_dist) AS avg_shot_distance
FROM gold_nba_shot_quality_features
GROUP BY 
  CASE
    WHEN is_close_shot = 1 THEN 'Close: <= 8 ft'
    WHEN is_mid_range_shot = 1 THEN 'Mid-range: >8 to <23.75 ft'
    WHEN is_three_point_shot = 1 THEN 'Three-point: >=23.75 to <28 ft'
    WHEN is_deep_shot = 1 THEN 'Deep: >=28 ft'
    ELSE 'Unknown'
  END
ORDER BY make_rate DESC
""").show()

In [0]:
# Make Rates by Defender Distance
spark.sql("""
SELECT 
  CASE
    WHEN is_very_tight_defense = 1 THEN 'Very Tight: <= 2 ft'
    WHEN is_tight_defense = 1 THEN 'Tight: >2 to <=4 ft'
    WHEN is_open_defense = 1 THEN 'Open: >4 to <=6 ft'
    WHEN is_wide_open_defense = 1 THEN 'Wide Open: >6 ft'
    ELSE 'Unknown'
  END AS defender_distance_bucket,
  COUNT(*) AS shots,
  AVG(label) AS make_rate,
  AVG(close_def_dist) AS avg_defender_distance
FROM gold_nba_shot_quality_features
GROUP BY 
  CASE
    WHEN is_very_tight_defense = 1 THEN 'Very Tight: <= 2 ft'
    WHEN is_tight_defense = 1 THEN 'Tight: >2 to <=4 ft'
    WHEN is_open_defense = 1 THEN 'Open: >4 to <=6 ft'
    WHEN is_wide_open_defense = 1 THEN 'Wide Open: >6 ft'
    ELSE 'Unknown'
  END
ORDER BY make_rate DESC
""").show()